In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from sklearn.metrics import f1_score
from torch_geometric.nn import GCNConv

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent

graph_path = ROOT / "data/processed/protein_function_graph_aac_score700_split.pt"

graph = torch.load(graph_path, weights_only=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
graph = graph.to(device)

print("Device:", device)
print(graph)

c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\rajdeep\Desktop\Capstone Project_GNN\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Device: cuda
Data(x=[18841, 22], edge_index=[2, 1067636], y=[18841, 96], edge_weight=[1067636], labeled_mask=[18841], train_mask=[18841], validation_mask=[18841], test_mask=[18841])


In [2]:
class GCNBaseline(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.30):
        super().__init__()

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, output_dim)

        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight):
        x = self.conv1(x, edge_index, edge_weight)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        return self.conv2(x, edge_index, edge_weight)

In [3]:
SEED = 42
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

INPUT_DIM = graph.x.shape[1]
HIDDEN_DIM = 64
OUTPUT_DIM = graph.y.shape[1]
DROPOUT = 0.30
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4

model = GCNBaseline(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    output_dim=OUTPUT_DIM,
    dropout=DROPOUT,
).to(device)

# Rare GO labels ko ignore hone se bachane ke liye class weighting.
train_targets = graph.y[graph.train_mask]
positive_count = train_targets.sum(dim=0)
negative_count = len(train_targets) - positive_count

pos_weight = negative_count / positive_count

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

print(model)
print("Positive-class weight range:")
print("min:", round(float(pos_weight.min()), 2))
print("max:", round(float(pos_weight.max()), 2))

GCNBaseline(
  (conv1): GCNConv(22, 64)
  (conv2): GCNConv(64, 96)
)
Positive-class weight range:
min: 7.66
max: 124.58


In [4]:
@torch.no_grad()
def evaluate(mask, threshold=0.5):
    model.eval()

    logits = model(graph.x, graph.edge_index, graph.edge_weight)
    loss = criterion(logits[mask], graph.y[mask])

    probabilities = torch.sigmoid(logits[mask]).cpu().numpy()
    targets = graph.y[mask].cpu().numpy()
    predictions = (probabilities >= threshold).astype(int)

    return {
        "loss": float(loss.item()),
        "micro_f1": f1_score(
            targets,
            predictions,
            average="micro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            targets,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

In [5]:
EPOCHS = 200
PATIENCE = 30

best_validation_loss = float("inf")
best_epoch = 0
best_state = None
epochs_without_improvement = 0

history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    logits = model(graph.x, graph.edge_index, graph.edge_weight)

    train_loss = criterion(
        logits[graph.train_mask],
        graph.y[graph.train_mask],
    )

    train_loss.backward()
    optimizer.step()

    validation_metrics = evaluate(graph.validation_mask)

    history.append(
        {
            "epoch": epoch,
            "train_loss": float(train_loss.item()),
            "validation_loss": validation_metrics["loss"],
            "validation_micro_f1": validation_metrics["micro_f1"],
            "validation_macro_f1": validation_metrics["macro_f1"],
        }
    )

    if validation_metrics["loss"] < best_validation_loss:
        best_validation_loss = validation_metrics["loss"]
        best_epoch = epoch
        epochs_without_improvement = 0

        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
    else:
        epochs_without_improvement += 1

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"train loss: {train_loss.item():.4f} | "
            f"val loss: {validation_metrics['loss']:.4f} | "
            f"val micro-F1: {validation_metrics['micro_f1']:.4f}"
        )

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation loss: {best_validation_loss:.4f}")

Epoch 001 | train loss: 1.3549 | val loss: 1.3338 | val micro-F1: 0.0636
Epoch 010 | train loss: 1.1848 | val loss: 1.1710 | val micro-F1: 0.1055
Epoch 020 | train loss: 1.0417 | val loss: 1.0472 | val micro-F1: 0.1191
Epoch 030 | train loss: 0.9846 | val loss: 1.0059 | val micro-F1: 0.1285
Epoch 040 | train loss: 0.9542 | val loss: 0.9871 | val micro-F1: 0.1340
Epoch 050 | train loss: 0.9374 | val loss: 0.9709 | val micro-F1: 0.1395
Epoch 060 | train loss: 0.9249 | val loss: 0.9602 | val micro-F1: 0.1411
Epoch 070 | train loss: 0.9149 | val loss: 0.9526 | val micro-F1: 0.1435
Epoch 080 | train loss: 0.9084 | val loss: 0.9479 | val micro-F1: 0.1463
Epoch 090 | train loss: 0.9033 | val loss: 0.9444 | val micro-F1: 0.1475
Epoch 100 | train loss: 0.8995 | val loss: 0.9417 | val micro-F1: 0.1472
Epoch 110 | train loss: 0.8962 | val loss: 0.9390 | val micro-F1: 0.1494
Epoch 120 | train loss: 0.8909 | val loss: 0.9358 | val micro-F1: 0.1494
Epoch 130 | train loss: 0.8920 | val loss: 0.9356 |

In [6]:
# Validation set par best probability threshold choose karenge.
threshold_candidates = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for threshold in threshold_candidates:
    metrics = evaluate(graph.validation_mask, threshold=threshold)
    threshold_results.append(
        {
            "threshold": round(float(threshold), 2),
            "validation_micro_f1": metrics["micro_f1"],
            "validation_macro_f1": metrics["macro_f1"],
        }
    )

threshold_df = pd.DataFrame(threshold_results)

best_threshold_row = threshold_df.loc[
    threshold_df["validation_micro_f1"].idxmax()
]

best_threshold = float(best_threshold_row["threshold"])

print("Chosen threshold:", best_threshold)
threshold_df

Chosen threshold: 0.8


,threshold,validation_micro_f1,validation_macro_f1
0,0.10,0.062360,0.064201
1,0.15,0.067677,0.071851
2,0.20,0.073664,0.080009
3,0.25,0.080456,0.089381
4,0.30,0.089360,0.100384
5,0.35,0.099743,0.112688
6,0.40,0.113412,0.127411
7,0.45,0.130873,0.144428
8,0.50,0.153422,0.163747
9,0.55,0.178915,0.182559


In [7]:
train_metrics = evaluate(graph.train_mask, threshold=best_threshold)
validation_metrics = evaluate(graph.validation_mask, threshold=best_threshold)
test_metrics = evaluate(graph.test_mask, threshold=best_threshold)

results_df = pd.DataFrame(
    [
        {"split": "train", **train_metrics},
        {"split": "validation", **validation_metrics},
        {"split": "test", **test_metrics},
    ]
)

results_df

,split,loss,micro_f1,macro_f1
0,train,0.869677,0.288854,0.240075
1,validation,0.929535,0.281071,0.230753
2,test,0.914137,0.272792,0.216405


In [8]:
models_dir = ROOT / "models"
results_dir = ROOT / "results"

models_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

model_path = models_dir / "gcn_aac_score700.pt"
metrics_path = results_dir / "gcn_aac_score700_metrics.csv"
history_path = results_dir / "gcn_aac_score700_training_history.csv"

torch.save(
    {
        "model_state_dict": best_state,
        "input_dim": INPUT_DIM,
        "hidden_dim": HIDDEN_DIM,
        "output_dim": OUTPUT_DIM,
        "dropout": DROPOUT,
        "best_epoch": best_epoch,
        "best_threshold": best_threshold,
    },
    model_path,
)

results_df.to_csv(metrics_path, index=False)
pd.DataFrame(history).to_csv(history_path, index=False)

print("Saved model:", model_path)
print("Saved metrics:", metrics_path)
print("Saved training history:", history_path)

Saved model: c:\Users\rajdeep\Desktop\Capstone Project_GNN\models\gcn_aac_score700.pt
Saved metrics: c:\Users\rajdeep\Desktop\Capstone Project_GNN\results\gcn_aac_score700_metrics.csv
Saved training history: c:\Users\rajdeep\Desktop\Capstone Project_GNN\results\gcn_aac_score700_training_history.csv
